<a href="https://colab.research.google.com/github/Naaish2/Convolutional-Variational-Autoencoder/blob/main/CVAE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
import keras
import tensorflow as tf
from keras import backend as k
from tensorflow.python.framework.ops import disable_eager_execution
disable_eager_execution()
k.clear_session()

Dataset loading and Pre-Processing

In [ ]:
# import minist dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
image_shape = (28, 28, 1)
latent_dim = 2
# change datatype and reshape data
x_train = x_train.astype('float32') / 255.
x_train = x_train.reshape((x_train.shape[0],) + image_shape)
x_test = x_test.astype('float32') / 255.
x_test = x_test.reshape((x_test.shape[0],) + image_shape)

Fetch each digit images

In [ ]:
# function to fetch 10 images of label 0 to 9
def get_images_1_to_10(x_train, y_train):
    selected_x, selected_y = [], []
    for i in range(10):
        number_index = np.where(y_train == i)[0]
        random_index = np.random.choice(len(number_index), 1, replace=False)
        select_index = number_index[random_index]
        selected_x.append(x_train[select_index[0]])
        selected_y.append(y_train[select_index][0])
    return np.array(selected_x, dtype="float32").reshape((len(selected_x),)+image_shape), np.array(selected_y, dtype="float32")


# select random 10 image of labeled 0 to 9
selected_x, selected_y = get_images_1_to_10(x_train, y_train)

Plot each digit original image

In [ ]:
def plot_image(selected_x, selected_y, title=None, save=None):
    ncols = selected_x.shape[0]
    fig, ax = plt.subplots(nrows=1, ncols=ncols, figsize=(20, 3))
    for x, y, ax_i in zip(selected_x, selected_y, ax):
        ax_i.imshow(x.reshape((28, 28)))
        ax_i.axis("off")
        ax_i.set_title(int(y))
    if title:
        fig.suptitle(title)
    plt.show()

plot_image(selected_x, selected_y)

Encoder Layer

In [ ]:
# Input
encoder_input = tf.keras.Input(shape=image_shape)
# convolutional layer 1
conv_1 = tf.keras.layers.Conv2D(
    filters=32, kernel_size=3, padding="same", activation="relu",)(encoder_input)

# convolutional layer 2
conv_2 = tf.keras.layers.Conv2D(filters=64,
                                kernel_size=3,
                                padding="same",
                                activation="relu",
                                )(conv_1)

# convolutional layer 3
conv_3 = tf.keras.layers.Conv2D(filters=64,
                                kernel_size=3,
                                padding="same",
                                activation="relu",
                                )(conv_2)
# Flatten the data
flatten = tf.keras.layers.Flatten()(conv_3)

# Dense layer 1
encoder_output = tf.keras.layers.Dense(128, activation="relu")(flatten)
z_mu = tf.keras.layers.Dense(latent_dim)(encoder_output)

# Dense layer 2
z_log_sigma = tf.keras.layers.Dense(latent_dim)(encoder_output)

Latent Layer

In [ ]:
# sampling function for latent layer
def sampling(args):
    z_mu, z_log_sigma = args

    # epsilon in simple normal distribution
    epsilon = k.random_normal(
        shape=(k.shape(z_mu)[0], latent_dim), mean=0., stddev=1.)
    return z_mu + k.exp(z_log_sigma) * epsilon


z = tf.keras.layers.Lambda(sampling, output_shape=(
    latent_dim,))([z_mu, z_log_sigma])

Decoder Layer

In [ ]:
# decoder layer
dense_2 = tf.keras.layers.Dense(128, activation="relu")
dense_3 = tf.keras.layers.Dense(np.prod(k.int_shape(conv_3)[1:]),
                                activation="relu"
                                )

# Reshape layer
reshape = tf.keras.layers.Reshape(k.int_shape(conv_3)[1:])

# Deconvolutional layer 1
conv_4 = tf.keras.layers.Conv2DTranspose(filters=64,
                                         kernel_size=3,
                                         padding="same",
                                         activation="relu"
                                         )
# Deconvolutional layer 2
conv_5 = tf.keras.layers.Conv2DTranspose(filters=64,
                                         kernel_size=3,
                                         padding="same",
                                         activation="relu"
                                         )

# Deconvolutional layer 3
conv_6 = tf.keras.layers.Conv2DTranspose(filters=32,
                                         kernel_size=3,
                                         padding="same",
                                         activation="relu"
                                         )

# convolutional layer 4
decoder_output = tf.keras.layers.Conv2D(filters=1,
                                        kernel_size=3,
                                        padding="same",
                                        activation="sigmoid"
                                        )

_dense_2 = dense_2(z)
_dense_3 = dense_3(_dense_2)
_reshape = reshape(_dense_3)
_conv_4 = conv_4(_reshape)
_conv_5 = conv_5(_conv_4)
_conv_6 = conv_6(_conv_5)
_decoder_output = decoder_output(_conv_6)

Defining loss function

In [ ]:
def vae_loss(x, z_decoded):
    x = k.flatten(x)
    z_decoded = k.flatten(z_decoded)

    # Reconstruction loss
    Reconstruction_loss = 786*keras.metrics.binary_crossentropy(x, z_decoded)

    # KL divergence
    kl_loss = -0.5 * k.mean(1 + z_log_sigma -
                            k.square(z_mu) - k.exp(z_log_sigma), axis=-1)
    return Reconstruction_loss + kl_loss

Visualizing model structure

In [ ]:
cvae = keras.Model(encoder_input, _decoder_output)
cvae.compile(optimizer='rmsprop', loss=vae_loss)
cvae.summary()

Train the CVAE Model

In [ ]:
cvae.fit(x=x_train, y=x_train,
         shuffle=True,
         epochs=10,
         batch_size=64)

Artificial Digit image Generations


In [ ]:
# select random 10 image of labeled 0 to 9 from test dataset
test_x, test_y =  get_images_1_to_10(x_test,y_test)

gen_x = cvae.predict(test_x)

plot_image(test_x,test_y, title="Origial unique Test Digits")
plot_image(gen_x,test_y,title="Artificial Generated Digits")